# CSI sur grille fine

Ce notebook calcule le CSI apres interpolation des predictions et de la verite terrain
sur une grille reguliere commune definie a partir du maillage fin.

Il garde seulement :
- l'evaluation horaire du CSI sur grille fine
- une figure explicite pour les 4 methodes comparees
- une vue qualitative binaire sur grille fine


In [ ]:
import math
import os
import pickle
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import ListedColormap

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.append(ROOT)

from python.create_dgl_dataset import TelemacDataset, TelemacDatasetWithQ, unpack_dynamic_sample
from python.eval_rollout import (
    build_model as _build_rollout_model,
    build_rollout_context,
    create_rollout_state,
    denormalize_state,
    load_model_checkpoint as _load_rollout_model_checkpoint,
    rollout_step,
)
from python.python_code.data_manip.extraction.telemac_file import TelemacFile
from python.python_code.data_manip.formats.regular_grid import interpolate_on_grid

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


In [ ]:
# =====================
# Parametres utilisateur
# =====================
DATA_DIRS = {
    'normal': '/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Mesh8_base.bin',
    'multimesh': '/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Multimesh_8_32.bin',
}
COARSE_MESH_PATHS = {
    'normal': '/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Mesh8_corrige.slf',
    'multimesh': '/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Mesh8_corrige.slf',
}
FINE_MESH_PATH = '/work/m24046/m24046mrcr/dataset_x8_avec_ts/short/maillage_3.slf'
DEFAULT_MESH = 'multimesh'

TEST_EVENTS = [
    'Group_1_peak_2600',
    'Group_2_peak_1000',
    'Group_2_peak_1200',
    'Group_2_peak_1600',
    'Group_4_peak_2000',
    'Group_1_peak_1200',
    'Group_1_peak_2400',
    'Group_3_peak_3400',
    'Group_1_peak_1400',
    'Group_1_peak_2000',
    'Group_1_peak_2200',
    'Group_2_peak_3600',
    'Group_3_peak_2200',
    'Group_3_peak_2800',
    'Group_4_peak_1200',
    'Group_4_peak_3000',
]

def build_dynamic_path(event_name):
    return (
        '/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/'
        f'{event_name}_{event_name}_0_35-64_interpolated.pkl'
    )

def build_hydro_path(event_name):
    return (
        '/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/'
        f'generated_hydrographs_{event_name}.liq'
    )

DYNAMIC_DIR = [build_dynamic_path(event_name) for event_name in TEST_EVENTS]
HYDRO_DIR = [build_hydro_path(event_name) for event_name in TEST_EVENTS]

METHODS = [
    {
        'name': 'Experiment 2',
        'ckpt_dir': '/work/m24046/m24046mrcr/paper/Experience7/Seed0/',
        'epoch': 900,
        'use_q_feature': True,
        'mesh': 'normal',
    },
    {
        'name': 'Experiment 3',
        'ckpt_dir': '/work/m24046/m24046mrcr/paper/Experience9/Seed0/',
        'epoch': 900,
        'use_q_feature': True,
        'mesh': 'normal',
    },
    {
        'name': 'Experiment 5',
        'ckpt_dir': '/work/m24046/m24046mrcr/paper/Experience8/Seed0/',
        'epoch': 900,
        'use_q_feature': True,
        'mesh': 'multimesh',
    },
    {
        'name': 'Experiment 6',
        'ckpt_dir': '/work/m24046/m24046mrcr/paper/Experience10/Seed0/',
        'epoch': 900,
        'use_q_feature': True,
        'mesh': 'multimesh',
    },
]
RUNS = {method['name']: f"{method['name']}@{method['epoch']}" for method in METHODS}
METHOD_BY_NAME = {method['name']: method for method in METHODS}

EXPERIENCE_COLORS = {
    'Experiment 2': '#ff7f0e',
    'Experiment 3': '#2ca02c',
    'Experiment 5': '#9467bd',
    'Experiment 6': '#8c564b',
}

DT_SECONDS = 1800.0
NUM_EDGE_FEATURES = 3
NUM_OUTPUT_FEATURES = 3
MP_LAYERS = 10
DO_CONCAT_TRICK = True
NUM_PROCESSOR_CHECKPOINT_SEGMENTS = 0

MAX_HOURS = 12
HOUR_IN_STEPS = max(1, int(round(3600.0 / DT_SECONDS)))
HORIZONS_STEPS = [hour * HOUR_IN_STEPS for hour in range(1, MAX_HOURS + 1)]
MAX_SEQUENCES = 205
THRESHOLD_M = 0.05
GRID_STEP_M = 25.0

QUAL_EVENT_INDEX = 7
QUAL_SEQUENCE_INDEX = 0
QUAL_HORIZON_HOURS = 6
QUAL_THRESHOLD_M = THRESHOLD_M
QUAL_STANDARD_METHOD = 'Experiment 3'
QUAL_MULTIMESH_METHOD = 'Experiment 6'


In [ ]:
def derive_fine_dynamic_files(dynamic_files):
    return [
        path.replace('/shortx8/', '/short/').replace('_interpolated.pkl', '.pkl')
        for path in dynamic_files
    ]

def build_model(num_input_features):
    return _build_rollout_model(
        num_input_features=num_input_features,
        num_edge_features=NUM_EDGE_FEATURES,
        num_output_features=NUM_OUTPUT_FEATURES,
        mp_layers=MP_LAYERS,
        do_concat_trick=DO_CONCAT_TRICK,
        num_processor_checkpoint_segments=NUM_PROCESSOR_CHECKPOINT_SEGMENTS,
    )

def build_dataset(sequence_length, ckpt_dir, use_q_feature, data_dir, dynamic_files, hydro_files):
    if use_q_feature:
        return TelemacDatasetWithQ(
            name='eval_test',
            data_dir=data_dir,
            dynamic_data_files=dynamic_files,
            hydro_data_files=hydro_files,
            split='test',
            ckpt_path=ckpt_dir,
            normalize=True,
            sequence_length=sequence_length,
            overlap=1,
            dt_seconds=DT_SECONDS,
        )

    return TelemacDataset(
        name='eval_test',
        data_dir=data_dir,
        dynamic_data_files=dynamic_files,
        split='test',
        ckpt_path=ckpt_dir,
        normalize=True,
        sequence_length=sequence_length,
        overlap=1,
    )

def load_dynamic_sequences(dynamic_files, sequence_length):
    sequences = []
    step = max(1, sequence_length - 1)
    for file_path in dynamic_files:
        with open(file_path, 'rb') as f:
            dynamic_data = pickle.load(f)
        for idx in range(0, len(dynamic_data) - sequence_length + 1, step):
            sequences.append(dynamic_data[idx:idx + sequence_length])
    return sequences

def csi_from_binary(pred_mask, gt_mask):
    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()
    denom = tp + fp + fn
    return float(tp / denom) if denom > 0 else math.nan

_MESH_CACHE = {}
_GRID_CACHE = {}
_MODEL_CACHE = {}

def get_mesh(mesh_path):
    if mesh_path not in _MESH_CACHE:
        _MESH_CACHE[mesh_path] = TelemacFile(mesh_path)
    return _MESH_CACHE[mesh_path]

def build_grid_from_step(mesh, grid_step_m):
    x = np.asarray(mesh.meshx[:mesh.npoin2], dtype=np.float64)
    y = np.asarray(mesh.meshy[:mesh.npoin2], dtype=np.float64)
    xmin, xmax = float(np.min(x)), float(np.max(x))
    ymin, ymax = float(np.min(y)), float(np.max(y))
    nx = int(np.ceil((xmax - xmin) / grid_step_m)) + 1
    ny = int(np.ceil((ymax - ymin) / grid_step_m)) + 1
    xi = xmin + np.arange(nx, dtype=np.float64) * grid_step_m
    yi = ymin + np.arange(ny, dtype=np.float64) * grid_step_m
    return np.meshgrid(xi, yi)

def get_common_grid(fine_mesh_path, grid_step_m):
    key = (fine_mesh_path, float(grid_step_m))
    if key not in _GRID_CACHE:
        fine_mesh = get_mesh(fine_mesh_path)
        grid = build_grid_from_step(fine_mesh, grid_step_m)
        trifinder = fine_mesh.tri.get_trifinder()
        domain_mask = trifinder(grid[0], grid[1]) == -1
        _GRID_CACHE[key] = (grid, domain_mask)
    return _GRID_CACHE[key]

def interpolate_depth_on_grid(tri, h_values, grid):
    h_grid, _ = interpolate_on_grid(tri, np.asarray(h_values, dtype=np.float64), grid=grid)
    return np.ma.asarray(h_grid)

def compute_grid_csi(h_pred_nodes, pred_mesh, h_gt_nodes, fine_mesh, grid, threshold, domain_mask):
    pred_grid = interpolate_depth_on_grid(pred_mesh.tri, h_pred_nodes, grid)
    gt_grid = interpolate_depth_on_grid(fine_mesh.tri, h_gt_nodes, grid)

    pred_masked = np.ma.asarray(pred_grid)
    gt_masked = np.ma.asarray(gt_grid)
    mask = np.ma.getmaskarray(pred_masked) | np.ma.getmaskarray(gt_masked) | domain_mask

    pred_vals = np.asarray(pred_masked.filled(np.nan))
    gt_vals = np.asarray(gt_masked.filled(np.nan))
    valid = (~mask) & np.isfinite(pred_vals) & np.isfinite(gt_vals)
    if not np.any(valid):
        return math.nan

    pred_binary = np.zeros_like(valid, dtype=bool)
    gt_binary = np.zeros_like(valid, dtype=bool)
    pred_binary[valid] = pred_vals[valid] >= threshold
    gt_binary[valid] = gt_vals[valid] >= threshold
    return csi_from_binary(pred_binary[valid], gt_binary[valid])

def get_cached_model(ds, method):
    key = (method['ckpt_dir'], method['epoch'], method['mesh'])
    if key not in _MODEL_CACHE:
        num_input_features = ds.base_graph.ndata['static'].shape[1] + 4
        model = build_model(num_input_features)
        _load_rollout_model_checkpoint(model, method['ckpt_dir'], method['epoch'], device=device)
        _MODEL_CACHE[key] = model
    return _MODEL_CACHE[key]

def evaluate_model_fine_grid(model, ds, fine_sequences, horizons_steps, coarse_mesh, fine_mesh, grid, domain_mask, max_sequences, threshold):
    rollout_context = build_rollout_context(ds, device=device, use_q_feature=True)
    agg = {step: [] for step in horizons_steps}
    step_set = set(horizons_steps)
    nseq = min(max_sequences, len(ds), len(fine_sequences))
    max_h = max(horizons_steps)

    for idx in range(nseq):
        graphs = ds[idx]
        fine_seq = fine_sequences[idx]
        rollout_state = create_rollout_state(graphs[0], rollout_context)

        for t in range(max_h):
            x_gt_full_n = graphs[t + 1].ndata['x'][:, rollout_context.dyn_start:rollout_context.dyn_start + rollout_context.dyn_len].to(device)
            x_gt = denormalize_state(x_gt_full_n[:, :3], rollout_context)
            q_t1_n = x_gt_full_n[:, 3:4]

            step_result = rollout_step(
                model,
                rollout_state,
                rollout_context,
                x_gt=x_gt,
                q_t1_n=q_t1_n,
            )

            step = t + 1
            if step in step_set:
                fine_x_t, fine_y_t, _ = unpack_dynamic_sample(fine_seq[t])
                fine_gt_t1 = np.asarray(fine_x_t, dtype=np.float64)[:, :3] + np.asarray(fine_y_t, dtype=np.float64)[:, :3]
                h_pred = step_result.predicted_state[:, 0].detach().cpu().numpy()
                h_gt_fine = fine_gt_t1[:, 0]
                agg[step].append(
                    compute_grid_csi(
                        h_pred_nodes=h_pred,
                        pred_mesh=coarse_mesh,
                        h_gt_nodes=h_gt_fine,
                        fine_mesh=fine_mesh,
                        grid=grid,
                        threshold=threshold,
                        domain_mask=domain_mask,
                    )
                )

            rollout_state = step_result.next_state

    summary = {}
    for step in horizons_steps:
        values = np.asarray(agg[step], dtype=float)
        if values.size == 0:
            continue
        summary[step] = {
            'csi_mean': float(np.nanmean(values)),
            'csi_std': float(np.nanstd(values)),
        }
    return summary

def evaluate_single_method_fine_grid(method, horizons_steps, max_sequences, threshold):
    sequence_length = max(horizons_steps) + 1
    dynamic_files = method.get('dynamic_dir', DYNAMIC_DIR)
    hydro_files = method.get('hydro_dir', HYDRO_DIR)
    fine_dynamic_files = method.get('fine_dynamic_dir', derive_fine_dynamic_files(dynamic_files))
    data_dir = method.get('data_dir', DATA_DIRS[method['mesh']])

    ds = build_dataset(
        sequence_length=sequence_length,
        ckpt_dir=method['ckpt_dir'],
        use_q_feature=method['use_q_feature'],
        data_dir=data_dir,
        dynamic_files=dynamic_files,
        hydro_files=hydro_files,
    )
    fine_sequences = load_dynamic_sequences(fine_dynamic_files, sequence_length=sequence_length)
    model = get_cached_model(ds, method)
    coarse_mesh = get_mesh(method.get('coarse_mesh_path', COARSE_MESH_PATHS[method['mesh']]))
    fine_mesh = get_mesh(method.get('fine_mesh_path', FINE_MESH_PATH))
    grid, domain_mask = get_common_grid(method.get('fine_mesh_path', FINE_MESH_PATH), method.get('grid_step_m', GRID_STEP_M))

    return evaluate_model_fine_grid(
        model=model,
        ds=ds,
        fine_sequences=fine_sequences,
        horizons_steps=horizons_steps,
        coarse_mesh=coarse_mesh,
        fine_mesh=fine_mesh,
        grid=grid,
        domain_mask=domain_mask,
        max_sequences=max_sequences,
        threshold=threshold,
    )

def evaluate_fixed_methods_fine_grid(methods, horizons_steps, max_sequences, threshold):
    fixed_metrics = {}
    for method in methods:
        run_name = f"{method['name']}@{method['epoch']}"
        fixed_metrics[run_name] = {
            'method': method['name'],
            'mesh': method['mesh'],
            'epoch': method['epoch'],
            'summary': evaluate_single_method_fine_grid(
                method,
                horizons_steps=horizons_steps,
                max_sequences=max_sequences,
                threshold=threshold,
            ),
        }
    return fixed_metrics


In [ ]:
grid, domain_mask = get_common_grid(FINE_MESH_PATH, GRID_STEP_M)
print(device)
print(f'Nombre de methodes: {len(METHODS)}')
print(f'HORIZONS_STEPS: {HORIZONS_STEPS}')
print(f'GRID_STEP_M: {GRID_STEP_M}')
print(f'GRID_SHAPE: {grid[0].shape}')
print(f'MASKED_OUTSIDE_DOMAIN: {int(np.count_nonzero(domain_mask))}')


In [ ]:
fixed_metrics_fine = evaluate_fixed_methods_fine_grid(
    METHODS,
    horizons_steps=HORIZONS_STEPS,
    max_sequences=MAX_SEQUENCES,
    threshold=THRESHOLD_M,
)


In [ ]:
# =====================
# Courbe CSI sur grille fine
# =====================
def _style_for_experience(label):
    return {
        'color': EXPERIENCE_COLORS[label],
        'marker': 'o',
        'linewidth': 2,
        'markersize': 5,
    }

def _plot_csi_curve(ax, hours, fixed_metrics, run_name, label):
    summary = fixed_metrics[run_name]['summary']
    mean_vals = np.array([summary.get(step, {}).get('csi_mean', np.nan) for step in HORIZONS_STEPS], dtype=float)
    std_vals = np.array([summary.get(step, {}).get('csi_std', np.nan) for step in HORIZONS_STEPS], dtype=float)
    lower = np.clip(mean_vals - std_vals, 0.0, 1.0)
    upper = np.clip(mean_vals + std_vals, 0.0, 1.0)
    style = _style_for_experience(label)
    ax.plot(hours, mean_vals, label=label, **style)
    ax.fill_between(hours, lower, upper, color=style['color'], alpha=0.18, linewidth=0)

def plot_fine_grid_csi(fixed_metrics):
    hours = np.array([step * DT_SECONDS / 3600.0 for step in HORIZONS_STEPS], dtype=float)
    fig, ax = plt.subplots(figsize=(12, 4.5))
    _plot_csi_curve(ax, hours, fixed_metrics, RUNS['Experiment 2'], 'Experiment 2')
    _plot_csi_curve(ax, hours, fixed_metrics, RUNS['Experiment 3'], 'Experiment 3')
    _plot_csi_curve(ax, hours, fixed_metrics, RUNS['Experiment 5'], 'Experiment 5')
    _plot_csi_curve(ax, hours, fixed_metrics, RUNS['Experiment 6'], 'Experiment 6')
    ax.set_title(f'Fine-grid CSI vs horizon (threshold={THRESHOLD_M:.3f} m)')
    ax.set_xlabel('Horizon (hours)')
    ax.set_ylabel('Fine-grid CSI')
    ax.set_ylim(0.0, 1.0)
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=False, ncol=4)
    fig.tight_layout()
    plt.show()

plot_fine_grid_csi(fixed_metrics_fine)


In [ ]:

# =====================
# Cartes binaires sur grille fine
# =====================
def hours_to_steps(hours):
    return int(round(hours * 3600.0 / DT_SECONDS))

def rollout_binary_map_for_method(method, event_index, sequence_index, horizon_step, threshold):
    dynamic_path = DYNAMIC_DIR[event_index]
    fine_dynamic_path = derive_fine_dynamic_files([dynamic_path])[0]
    hydro_files = [HYDRO_DIR[event_index]]
    sequence_length = horizon_step + 1

    ds = build_dataset(
        sequence_length=sequence_length,
        ckpt_dir=method['ckpt_dir'],
        use_q_feature=method['use_q_feature'],
        data_dir=method.get('data_dir', DATA_DIRS[method['mesh']]),
        dynamic_files=[dynamic_path],
        hydro_files=hydro_files,
    )
    fine_sequences = load_dynamic_sequences([fine_dynamic_path], sequence_length=sequence_length)
    model = get_cached_model(ds, method)
    coarse_mesh = get_mesh(method.get('coarse_mesh_path', COARSE_MESH_PATHS[method['mesh']]))
    fine_mesh_path = method.get('fine_mesh_path', FINE_MESH_PATH)
    fine_mesh = get_mesh(fine_mesh_path)
    grid, domain_mask = get_common_grid(fine_mesh_path, method.get('grid_step_m', GRID_STEP_M))

    graphs = ds[sequence_index]
    fine_seq = fine_sequences[sequence_index]
    rollout_context = build_rollout_context(ds, device=device, use_q_feature=True)
    rollout_state = create_rollout_state(graphs[0], rollout_context)
    predicted_state = None

    for t in range(horizon_step):
        x_gt_full_n = graphs[t + 1].ndata['x'][:, rollout_context.dyn_start:rollout_context.dyn_start + rollout_context.dyn_len].to(device)
        x_gt = denormalize_state(x_gt_full_n[:, :3], rollout_context)
        q_t1_n = x_gt_full_n[:, 3:4]
        step_result = rollout_step(
            model,
            rollout_state,
            rollout_context,
            x_gt=x_gt,
            q_t1_n=q_t1_n,
        )
        predicted_state = step_result.predicted_state
        rollout_state = step_result.next_state

    fine_x_t, fine_y_t, _ = unpack_dynamic_sample(fine_seq[horizon_step - 1])
    fine_gt_t1 = np.asarray(fine_x_t, dtype=np.float64)[:, :3] + np.asarray(fine_y_t, dtype=np.float64)[:, :3]

    h_pred_nodes = predicted_state[:, 0].detach().cpu().numpy()
    h_gt_nodes = fine_gt_t1[:, 0]
    pred_grid = interpolate_depth_on_grid(coarse_mesh.tri, h_pred_nodes, grid)
    gt_grid = interpolate_depth_on_grid(fine_mesh.tri, h_gt_nodes, grid)

    pred_masked = np.ma.asarray(pred_grid)
    gt_masked = np.ma.asarray(gt_grid)
    mask = np.ma.getmaskarray(pred_masked) | np.ma.getmaskarray(gt_masked) | domain_mask

    pred_vals = np.asarray(pred_masked.filled(np.nan))
    gt_vals = np.asarray(gt_masked.filled(np.nan))
    valid = (~mask) & np.isfinite(pred_vals) & np.isfinite(gt_vals)
    pred_binary = np.zeros_like(valid, dtype=bool)
    gt_binary = np.zeros_like(valid, dtype=bool)
    pred_binary[valid] = pred_vals[valid] >= threshold
    gt_binary[valid] = gt_vals[valid] >= threshold

    return {
        'event_name': TEST_EVENTS[event_index],
        'reference_binary': np.ma.array(gt_binary.astype(float), mask=mask),
        'prediction_binary': np.ma.array(pred_binary.astype(float), mask=mask),
        'csi': csi_from_binary(pred_binary[valid], gt_binary[valid]) if np.any(valid) else math.nan,
    }

def plot_qualitative_fine_grid_maps():
    horizon_step = hours_to_steps(QUAL_HORIZON_HOURS)
    reference_and_standard = rollout_binary_map_for_method(
        METHOD_BY_NAME[QUAL_STANDARD_METHOD],
        event_index=QUAL_EVENT_INDEX,
        sequence_index=QUAL_SEQUENCE_INDEX,
        horizon_step=horizon_step,
        threshold=QUAL_THRESHOLD_M,
    )
    multimesh_payload = rollout_binary_map_for_method(
        METHOD_BY_NAME[QUAL_MULTIMESH_METHOD],
        event_index=QUAL_EVENT_INDEX,
        sequence_index=QUAL_SEQUENCE_INDEX,
        horizon_step=horizon_step,
        threshold=QUAL_THRESHOLD_M,
    )

    cmap = ListedColormap(['white', '#1f4e79'])
    cmap.set_bad(color='#d9d9d9')
    fig, axes = plt.subplots(1, 3, figsize=(12.6, 4.8), sharex=True, sharey=True)

    axes[0].imshow(reference_and_standard['reference_binary'], origin='lower', cmap=cmap, vmin=0, vmax=1)
    axes[0].set_title('Reference')
    axes[0].set_axis_off()

    axes[1].imshow(reference_and_standard['prediction_binary'], origin='lower', cmap=cmap, vmin=0, vmax=1)
    axes[1].set_title(f"{QUAL_STANDARD_METHOD}\nCSI={reference_and_standard['csi']:.3f}")
    axes[1].set_axis_off()

    axes[2].imshow(multimesh_payload['prediction_binary'], origin='lower', cmap=cmap, vmin=0, vmax=1)
    axes[2].set_title(f"{QUAL_MULTIMESH_METHOD}\nCSI={multimesh_payload['csi']:.3f}")
    axes[2].set_axis_off()

    fig.suptitle(
        f"{reference_and_standard['event_name']} | horizon={QUAL_HORIZON_HOURS} h | threshold={100 * QUAL_THRESHOLD_M:.0f} cm",
        y=0.98,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

plot_qualitative_fine_grid_maps()
